# Assignment 3 — Fine-Tuning DistilBERT for Genre Classification



In [16]:
import os

!pip install -q --upgrade transformers datasets evaluate accelerate peft
!pip install -q huggingface_hub wandb scikit-learn seaborn matplotlib pandas gdown
print("All packages installed")

All packages installed


In [ ]:
import os, random, json, gzip, requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from collections import defaultdict

HF_USERNAME   = "Saumya3007"
WANDB_API_KEY = "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"
MODEL_NAME    = "distilbert-base-uncased"
HF_REPO       = f"Saumya3007/distilbert-goodreads-genre"

GENRES = ["children","comics_graphic","fantasy_paranormal",
          "history_biography","mystery_thriller_crime",
          "poetry","romance","young_adult"]
LABEL2ID = {g: i for i, g in enumerate(GENRES)}
ID2LABEL = {i: g for i, g in enumerate(GENRES)}
NUM_LABELS = len(GENRES)

MAX_LENGTH       = 512
BATCH_SIZE       = 16 if torch.cuda.is_available() else 8
EPOCHS           = 3
LR               = 5e-5
WARMUP_STEPS     = 100
WEIGHT_DECAY     = 0.01
HEAD_PER_GENRE   = 10000
SAMPLE_PER_GENRE = 1000
TRAIN_SPLIT      = 0.8
SEED             = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
os.makedirs("results", exist_ok=True)
os.makedirs("src",     exist_ok=True)

print(f"Device  : {DEVICE}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"HF Repo : {HF_REPO}  |  Batch: {BATCH_SIZE}  |  Epochs: {EPOCHS}")


In [3]:
import wandb
from huggingface_hub import login
login()
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY, relogin=True)
print("Logged in")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pancholisaumya (pancholisaumya-iit) to https://api.wandb.ai. Use `wandb login

Logged in


In [4]:
GENRE_URLS = {
    "poetry":                 "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz",
    "children":               "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz",
    "comics_graphic":         "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz",
    "fantasy_paranormal":     "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz",
    "history_biography":      "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz",
    "mystery_thriller_crime": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz",
    "romance":                "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz",
    "young_adult":            "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz",
}

def load_reviews(url, head=HEAD_PER_GENRE, sample_size=SAMPLE_PER_GENRE):
    reviews, count = [], 0
    response = requests.get(url, stream=True)
    with gzip.open(response.raw, "rt", encoding="utf-8") as f:
        for line in f:
            reviews.append(json.loads(line)["review_text"])
            count += 1
            if head and count >= head:
                break
    return random.sample(reviews, min(sample_size, len(reviews)))

genre_reviews = {}
for genre, url in GENRE_URLS.items():
    print(f"Loading {genre} ...", end=" ")
    genre_reviews[genre] = load_reviews(url)
    print(f"{len(genre_reviews[genre])}")
print("Dataset loaded")


Loading poetry ... 1000
Loading children ... 1000
Loading comics_graphic ... 1000
Loading fantasy_paranormal ... 1000
Loading history_biography ... 1000
Loading mystery_thriller_crime ... 1000
Loading romance ... 1000
Loading young_adult ... 1000
Dataset loaded


In [5]:
counts  = {g: len(r) for g, r in genre_reviews.items()}
lengths = [len(str(r).split()) for reviews in genre_reviews.values() for r in reviews]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = plt.cm.Purples(np.linspace(0.4, 0.9, len(counts)))
axes[0].barh(list(counts.keys()), list(counts.values()), color=colors)
axes[0].set_xlabel("Count"); axes[0].set_title("Genre Distribution"); axes[0].invert_yaxis()
axes[1].hist(lengths, bins=60, color="#7B2D8B", alpha=0.7, edgecolor="white")
axes[1].axvline(np.mean(lengths), color="red", linestyle="--", label=f"Mean={np.mean(lengths):.0f}")
axes[1].set_xlabel("Words"); axes[1].set_ylabel("Count")
axes[1].set_title("Review Length Distribution"); axes[1].legend()
plt.tight_layout()
plt.savefig("results/dataset_eda.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Avg length: {np.mean(lengths):.0f} words  |  Max: {max(lengths)}")


Avg length: 121 words  |  Max: 1864


In [6]:
train_texts, train_labels, test_texts, test_labels = [], [], [], []
for genre, reviews in genre_reviews.items():
    cutoff = int(len(reviews) * TRAIN_SPLIT)
    train_texts += reviews[:cutoff]; train_labels += [genre] * cutoff
    test_texts  += reviews[cutoff:]; test_labels  += [genre] * (len(reviews) - cutoff)
print(f"Train: {len(train_texts)}  |  Test: {len(test_texts)}")


Train: 6400  |  Test: 1600


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_texts)
X_test  = vectorizer.transform(test_texts)
lr_preds = LogisticRegression(max_iter=1000).fit(X_train, train_labels).predict(X_test)
print("TF-IDF Baseline:")
print(classification_report(test_labels, lr_preds))


TF-IDF Baseline:
                        precision    recall  f1-score   support

              children       0.67      0.63      0.65       200
        comics_graphic       0.76      0.72      0.74       200
    fantasy_paranormal       0.38      0.30      0.34       200
     history_biography       0.56      0.53      0.54       200
mystery_thriller_crime       0.51      0.52      0.51       200
                poetry       0.62      0.77      0.68       200
               romance       0.57      0.60      0.58       200
           young_adult       0.43      0.45      0.44       200

              accuracy                           0.56      1600
             macro avg       0.56      0.56      0.56      1600
          weighted avg       0.56      0.56      0.56      1600



In [8]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
test_enc  = tokenizer(test_texts,  truncation=True, padding=True, max_length=MAX_LENGTH)

train_labels_enc = [LABEL2ID[y] for y in train_labels]
test_labels_enc  = [LABEL2ID[y] for y in test_labels]

class GoodreadsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings; self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self): return len(self.labels)

train_dataset = GoodreadsDataset(train_enc, train_labels_enc)
test_dataset  = GoodreadsDataset(test_enc,  test_labels_enc)
print(f"Train: {len(train_dataset)}  |  Test: {len(test_dataset)}")


Train: 6400  |  Test: 1600


In [9]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS).to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {next(model.parameters()).device}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Params: 66,959,624
Device: cuda:0


In [10]:
wandb.init(
    project="hf-docker-assignment", name=f"distilbert-{EPOCHS}ep",
    config={"model": MODEL_NAME, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
            "lr": LR, "max_length": MAX_LENGTH, "seed": SEED,
            "train_samples": len(train_dataset), "test_samples": len(test_dataset)}
)
print(f"WandB: {wandb.run.url}")


WandB: https://wandb.ai/pancholisaumya-iit/hf-docker-assignment/runs/2m6m9ude


In [11]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred[0], axis=-1)
    return {"accuracy": accuracy_score(eval_pred[1], preds)}

training_args = TrainingArguments(
    output_dir="results/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    eval_strategy="steps", eval_steps=100,
    save_strategy="steps", save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_dir="results/logs", logging_steps=100,
    report_to="wandb",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Train samples: {len(train_dataset)}  |  FP16: {torch.cuda.is_available()}")
print("Starting training ...")
result = trainer.train()
print(f"Training complete!  Loss: {result.training_loss:.4f}")
wandb.log({"final_train_loss": result.training_loss})


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Train samples: 6400  |  FP16: True
Starting training ...


Step,Training Loss,Validation Loss,Accuracy
100,1.941479,1.548454,0.501250
200,1.393613,1.276883,0.542500
300,1.269718,1.225616,0.565625
400,1.246826,1.186937,0.594375
500,1.002574,1.195712,0.591250
600,0.919082,1.207605,0.587500
700,0.906018,1.164386,0.600000
800,0.891810,1.148208,0.612500
900,0.588186,1.191289,0.610000
1000,0.605448,1.202775,0.615625


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training complete!  Loss: 0.9968


In [12]:
logs       = trainer.state.log_history
train_logs = [l for l in logs if "loss" in l and "eval_loss" not in l]
eval_logs  = [l for l in logs if "eval_loss" in l]

if train_logs and eval_logs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot([l["step"] for l in train_logs], [l["loss"] for l in train_logs], "b-", lw=2)
    axes[0].set_title("Training Loss"); axes[0].set_xlabel("Steps"); axes[0].grid(alpha=0.3)
    axes[1].plot([l["step"] for l in eval_logs], [l["eval_loss"] for l in eval_logs], "r-", lw=2)
    ax2 = axes[1].twinx()
    ax2.plot([l["step"] for l in eval_logs], [l.get("eval_accuracy", 0) for l in eval_logs], "g--", lw=2, label="Accuracy")
    axes[1].set_title("Val Loss / Accuracy"); axes[1].set_xlabel("Steps"); axes[1].grid(alpha=0.3)
    ax2.legend(loc="upper right")
    plt.tight_layout()
    plt.savefig("results/training_history.png", dpi=150, bbox_inches="tight")
    wandb.log({"training_history": wandb.Image("results/training_history.png")})
    plt.show()


In [13]:
from sklearn.metrics import classification_report, confusion_matrix

print("Evaluating ...")
local_metrics    = trainer.evaluate(test_dataset)
predicted_results = trainer.predict(test_dataset)
predicted_labels  = [ID2LABEL[i] for i in predicted_results.predictions.argmax(-1).flatten()]

print("LOCAL MODEL RESULTS")
for k, v in local_metrics.items(): print(f"  {k}: {v:.4f}")
print()
print(classification_report(test_labels, predicted_labels))

with open("results/local_metrics.json", "w") as f:
    json.dump(local_metrics, f, indent=2)
wandb.log({k: v for k, v in local_metrics.items()})


Evaluating ...


LOCAL MODEL RESULTS
  eval_loss: 1.2028
  eval_accuracy: 0.6156
  eval_runtime: 6.7234
  eval_samples_per_second: 237.9740
  eval_steps_per_second: 14.8730
  epoch: 3.0000

                        precision    recall  f1-score   support

              children       0.69      0.74      0.71       200
        comics_graphic       0.86      0.76      0.81       200
    fantasy_paranormal       0.43      0.53      0.47       200
     history_biography       0.61      0.61      0.61       200
mystery_thriller_crime       0.61      0.57      0.59       200
                poetry       0.77      0.79      0.78       200
               romance       0.63      0.56      0.59       200
           young_adult       0.37      0.36      0.37       200

              accuracy                           0.62      1600
             macro avg       0.62      0.62      0.62      1600
          weighted avg       0.62      0.62      0.62      1600



In [17]:
genre_cls = defaultdict(int)
for t, p, in zip(test_labels, predicted_labels): genre_cls[(t, p)] += 1
df_wide = (pd.DataFrame([{"True Genre": t, "Predicted Genre": p, "Count": c}
                          for (t, p), c in genre_cls.items()])
           .pivot_table(index="True Genre", columns="Predicted Genre", values="Count", fill_value=0))
plt.figure(figsize=(10, 8))
sns.set(style="ticks", font_scale=1.1)
sns.heatmap(df_wide, linewidths=1, cmap="Purples", annot=True, fmt=".0f")
plt.xticks(rotation=45, ha="right")
plt.title("Confusion Heatmap — Local Model"); plt.tight_layout()
plt.savefig("results/cm_local.png", dpi=150, bbox_inches="tight")
wandb.log({"cm_local": wandb.Image("results/cm_local.png")}); plt.show()

In [18]:
correct = [(t, p, tx) for t, p, tx in zip(test_labels, predicted_labels, test_texts) if t == p]
wrong   = [(t, p, tx) for t, p, tx in zip(test_labels, predicted_labels, test_texts) if t != p]

print("CORRECTLY CLASSIFIED")
for t, p, tx in random.sample(correct, min(5, len(correct))):
    print(f"LABEL: {t}"); print(f"TEXT : {str(tx)[:120]} ...\n")

print("MISCLASSIFIED")
for t, p, tx in random.sample(wrong, min(5, len(wrong))):
    print(f"TRUE: {t}  PREDICTED: {p}"); print(f"TEXT: {str(tx)[:120]} ...\n")


CORRECTLY CLASSIFIED
LABEL: history_biography
TEXT : I doubt there's ever been a more completed and detailed book on the history of anything. ...

LABEL: poetry
TEXT : The Antigone Poems warrants multiple rereads and it isn't easy going; it's intense, vocal and anguished. 
 This is an im ...

LABEL: poetry
TEXT : l ynbGy wSf m nthytu mnh lltw sh`ran , flwSf l'dq lh 'nWh khwTr , w lyst khwTr fqT , bl hy khwTrmhlhl@ ! 
 'r~ 'n mjrWd  ...

LABEL: mystery_thriller_crime
TEXT : Journalist McKenna Jordan is chasing down her latest story when things get personal and a past mystery is suddenly front ...

LABEL: children
TEXT : I wonder actually how old are the grandpas and grandmas of Charlie? In Charlie and the Great Glass Elevator, Grandpa Joe ...

MISCLASSIFIED
TRUE: history_biography  PREDICTED: mystery_thriller_crime
TEXT: I registered a book at BookCrossing.com! 
 http://www.BookCrossing.com/journal/12840277 ...

TRUE: fantasy_paranormal  PREDICTED: history_biography
TEXT: delightful! ..

## STEP 15 — Save Model Locally

In [19]:
LOCAL_DIR = "results/saved_model"
trainer.save_model(LOCAL_DIR)
tokenizer.save_pretrained(LOCAL_DIR)
print(f"Saved to {LOCAL_DIR}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to results/saved_model


In [20]:
from huggingface_hub import HfApi

acc  = local_metrics.get("eval_accuracy", 0)
loss = local_metrics.get("eval_loss", 0)

model_card = f"""---
language: en
license: apache-2.0
tags: [text-classification, distilbert, goodreads]
---
# DistilBERT Goodreads Genre Classifier
Fine-tuned `distilbert-base-uncased` on 8-genre Goodreads book reviews.
## Labels: {", ".join(GENRES)}
## Results: Accuracy={acc:.4f}  Loss={loss:.4f}
## WandB: https://api.wandb.ai/links/pancholisaumya-iit/z7mxf7zr
"""
with open(f"{LOCAL_DIR}/README.md", "w") as f: f.write(model_card)

print(f"Pushing to {HF_REPO} ...")
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
HfApi().upload_file(path_or_fileobj=f"{LOCAL_DIR}/README.md",
                    path_in_repo="README.md", repo_id=HF_REPO, repo_type="model")
print(f"Model live at https://huggingface.co/{HF_REPO}")


Pushing to Saumya3007/distilbert-goodreads-genre ...


README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eu62qc2/model.safetensors:   0%|          |  574kB /  268MB            

Model live at https://huggingface.co/Saumya3007/distilbert-goodreads-genre


In [21]:
hf_tokenizer = DistilBertTokenizerFast.from_pretrained(HF_REPO)
hf_model     = DistilBertForSequenceClassification.from_pretrained(HF_REPO)

hf_trainer = Trainer(
    model=hf_model,
    args=TrainingArguments(output_dir="results/hf_eval", report_to="none",
                           per_device_eval_batch_size=BATCH_SIZE),
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)
hf_metrics = hf_trainer.evaluate()
print("HUGGINGFACE MODEL RESULTS")
for k, v in hf_metrics.items(): print(f"  {k}: {v:.4f}")
with open("results/hf_metrics.json", "w") as f: json.dump(hf_metrics, f, indent=2)
wandb.log({f"hf_{k}": v for k, v in hf_metrics.items()})


tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

HUGGINGFACE MODEL RESULTS
  eval_loss: 1.2028
  eval_model_preparation_time: 0.0048
  eval_accuracy: 0.6156
  eval_runtime: 30.2779
  eval_samples_per_second: 52.8440
  eval_steps_per_second: 3.3030


In [22]:
lv = local_metrics.get("eval_accuracy", 0)
hv = hf_metrics.get("eval_accuracy", 0)

x, w = np.arange(2), 0.35
fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - w/2, [lv, local_metrics.get("eval_loss", 0)], w, label="Local",       color="#5b2c8d")
b2 = ax.bar(x + w/2, [hv, hf_metrics.get("eval_loss", 0)],    w, label="HuggingFace", color="#a78ec5")
ax.set_xticks(x); ax.set_xticklabels(["Accuracy", "Loss"])
ax.set_title("Local vs HuggingFace Hub"); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.bar_label(b1, fmt="%.4f", padding=3, fontsize=9)
ax.bar_label(b2, fmt="%.4f", padding=3, fontsize=9)
plt.tight_layout()
plt.savefig("results/comparison.png", dpi=150, bbox_inches="tight")
wandb.log({"comparison": wandb.Image("results/comparison.png")}); plt.show()

print(f"{'Metric':<22} {'Local':>10} {'HF Hub':>10} {'Delta':>10}")
print("─" * 54)
for k in ["eval_accuracy", "eval_loss"]:
    print(f"{k:<22} {local_metrics.get(k,0):>10.4f} {hf_metrics.get(k,0):>10.4f} {abs(local_metrics.get(k,0)-hf_metrics.get(k,0)):>10.4f}")


Metric                      Local     HF Hub      Delta
──────────────────────────────────────────────────────
eval_accuracy              0.6156     0.6156     0.0000
eval_loss                  1.2028     1.2028     0.0000


In [25]:
import shutil
if wandb.run: wandb.finish()
shutil.make_archive("assignment3_results", "zip", "results")
try:
    from google.colab import files
    files.download("assignment3_results.zip")
    print("Downloading results zip ...")
except:
    print("Results saved to ./results/")
print(f"Model : https://huggingface.co/{HF_REPO}")
print(f"WandB : https://api.wandb.ai/links/pancholisaumya-iit/z7mxf7zr")


epoch,▁
eval/accuracy,▁▄▅▇▇▆▇██████
eval/loss,█▃▂▂▂▂▁▁▂▂▂▂▂
eval/runtime,▁█▆█▇█████▇█▅
eval/samples_per_second,█▁▃▁▂▁▁▁▁▁▂▁▄
eval/steps_per_second,█▁▃▁▂▁▁▁▁▁▂▁▄
eval_accuracy,▁
eval_loss,▁
eval_runtime,▁
eval_samples_per_second,▁
+18,...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model : https://huggingface.co/Saumya3007/distilbert-goodreads-genre
WandB : https://api.wandb.ai/links/pancholisaumya-iit/z7mxf7zr


Docker Commands (Task 2 & Task 9)

### Dev Image (train + eval)
```bash
docker build -t assignment3-dev -f Dockerfile .
docker run --rm \
  -e HF_TOKEN=<token> -e WANDB_API_KEY=<key> \
  -v $(pwd)/results:/app/results \
  assignment3-dev python -m src.train
```

### Prod Image (eval only — pulls from HF Hub)
```bash
docker build -t assignment3-prod -f Dockerfile.prod .
docker run --rm \
  -e HF_TOKEN=<token> \
  -e HF_REPO=Saumya3007/distilbert-goodreads-genre \
  -v $(pwd)/results:/home/appuser/app/results \
  assignment3-prod
```
